# scraping our bill text

In [6]:
import requests # for making http (web) requests
import pandas as pd # for working with tabular (spreadsheet) data
import csv # also for working with tabular data, in csv format

# this grabs the CSV from the previous section. If you get a file
# not found error make sure you go through the previous section to 
# save that csv
bills = pd.read_csv('congress_clean.csv')

df = pd.DataFrame(bills)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Unnamed: 0               148 non-null    int64  
 1   Legislation Number       148 non-null    object 
 2   URL                      148 non-null    object 
 3   Congress                 148 non-null    object 
 4   Title                    146 non-null    object 
 5   Sponsor                  148 non-null    object 
 6   Party of Sponsor         148 non-null    object 
 7   Date of Introduction     115 non-null    object 
 8   Committees               114 non-null    object 
 9   Latest Action            146 non-null    object 
 10  Latest Action Date       146 non-null    object 
 11  Latest Summary           61 non-null     object 
 12  Amends Bill              33 non-null     object 
 13  Date Offered             31 non-null     object 
 14  Date Submitted           2

In [8]:
df = df.drop_duplicates(subset=['Legislation Number'], keep='first')

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 148 entries, 0 to 147
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Unnamed: 0               148 non-null    int64  
 1   Legislation Number       148 non-null    object 
 2   URL                      148 non-null    object 
 3   Congress                 148 non-null    object 
 4   Title                    146 non-null    object 
 5   Sponsor                  148 non-null    object 
 6   Party of Sponsor         148 non-null    object 
 7   Date of Introduction     115 non-null    object 
 8   Committees               114 non-null    object 
 9   Latest Action            146 non-null    object 
 10  Latest Action Date       146 non-null    object 
 11  Latest Summary           61 non-null     object 
 12  Amends Bill              33 non-null     object 
 13  Date Offered             31 non-null     object 
 14  Date Submitted           2

In order to scrape the bill text, we need just the bill number. In order to get that, we need to go through the `Legislation Number` column and extract just the number.

In [13]:
df['Legislation Number']

0        H.R. 1112
1           S. 435
2       H.Res. 886
3       S.Res. 464
4       H.Res. 269
          ...     
143    H.Amdt. 195
144    H.Amdt. 193
145    H.Amdt. 256
146    H.Amdt. 257
147    H.Amdt. 255
Name: Legislation Number, Length: 148, dtype: object

In [12]:
# we can use the split() method to split up the single string
# into two strings, by the empty space in between them

bill = "H.R. 1112"
bill.split(' ')

['H.R.', '1112']

In [14]:
# we can write a for loop to append the number to a list
# involves checking if the item is a number, using "isnumeric"

numbers = []
for item in bill.split(' '):
    if item.isnumeric():
        numbers.append(item)

In [15]:
item

'1112'

Now we will write a loop that:
- goes through each row of `df['Legislation Number']`
- turns that row into a string, using `str()` function 
- splits that row by the empty space using `split()`
- writes another loop to check if the item `isnumeric()`
- appends the numeric item to a new list

In [16]:
## go through each row in numbers column of our spreadsheet
## extract the number and put into a separate list
numbers = []
for row in df['Legislation Number']:
    # need to change data type to string in order to use split()
    row = str(row)
    splitted = row.split(' ')
    for item in splitted:
        if item.isnumeric():
            numbers.append(item)

## functions

How do you write a function?

```
def add(x, y):
    print(x + y)
```

How do you call a function?

```
add(5, 7)
```

In [17]:
# function that contains a loop to insert bill numbers
# into the URL, then to grab the content and add to a new list
def scrape_bill_text(numbers):
    bills_text = []
    for item in numbers:
        # make sure to put the right congress session
        url = (f'https://www.congress.gov/117/bills/hr{item}/BILLS-117hr{item}ih.htm')
        response = requests.get(url)
        content = response.content
        bills_text.append(content)
    return bills_text

In [18]:
# full_text = scrape_just_text(numbers)
sample = scrape_bill_text(numbers[:10])

In [20]:
len(sample)

10

In [23]:
sample[0]

b"<html><body><pre>\n[Congressional Bills 117th Congress]\n[From the U.S. Government Publishing Office]\n[H.R. 1112 Introduced in House (IH)]\n\n&lt;DOC&gt;\n\n\n\n\n\n\n117th CONGRESS\n  1st Session\n                                H. R. 1112\n\n   To require a report on the military coup in Burma, and for other \n                               purposes.\n\n\n_______________________________________________________________________\n\n\n                    IN THE HOUSE OF REPRESENTATIVES\n\n                           February 18, 2021\n\n    Mr. Connolly (for himself, Mr. Price of North Carolina, and Mr. \n  Buchanan) introduced the following bill; which was referred to the \n                      Committee on Foreign Affairs\n\n_______________________________________________________________________\n\n                                 A BILL\n\n\n \n   To require a report on the military coup in Burma, and for other \n                               purposes.\n\n    Be it enacted by the 

In [25]:
## still need to decode and then save.